# CO3133 Assignment 3 — M2-only GPU reproduction

This notebook runs ONLY the locked M1 `filtered_original` pipeline and nine M2 dev experiments. No M3, held-out test evaluation, slides or video. Use the pinned dataset and keep raw images/text private.

In [ ]:
import subprocess, pathlib, sys, os
print("Python",sys.version)
subprocess.run(["nvidia-smi"],check=False)
print("Make sure runtime has enough RAM/disk (~3–5 GB working storage).")

## 1. Clone published M2 repository

Clone the public repository after the M2 patch has been pushed. The original M1 locked files must remain unchanged.

In [ ]:
import subprocess, os
from pathlib import Path
subprocess.run(["git","clone","https://github.com/volethanhtrieu/DL_Assignment-03.git","/content/DL_Assignment-03"], check=True)
os.chdir("/content/DL_Assignment-03")
print("Working directory:", os.getcwd())

In [ ]:
import subprocess,sys
subprocess.run([sys.executable,"-m","pip","install","-r","m2m3/requirements.txt","-q"],check=True)
subprocess.run([sys.executable,"-m","pytest","-q","m2m3/tests"],check=True)
subprocess.run([sys.executable,"m2m3/experiment.py","check","--repo","."],check=True)

## 2. Real data and feature extraction

Download 4 **SHA-256-pinned** Parquet shards, then extract frozen ResNet-18/DistilBERT features. **This stage can be long**; the time and tokenizer stats are saved to `features/primary/metadata.json`. Do not switch dataset revisions if a download fails.

In [ ]:
import subprocess,sys
subprocess.run([sys.executable,"m2m3/download_mirror.py","--repo",".","--out","data/mirror"],check=True)
subprocess.run([sys.executable,"m2m3/experiment.py","extract","--repo",".","--mirror","data/mirror","--out","features/primary","--batch-size","16","--device","auto"],check=True)

## 3. M2 draft — 9 REAL development runs

Image-only, text-only, concat × seeds 42/43/44. **No held-out test evaluation here.** The summary describes dev results, not generalization on test.

In [ ]:
subprocess.run([sys.executable,"m2m3/run_all.py","--repo",".","--features","features/primary","--runs","runs/primary","--milestone","m2"],check=True)
subprocess.run([sys.executable,"m2m3/summarize_dev.py","--runs","runs/primary","--out","results/m2_dev","--milestone","m2"],check=True)
print(Path("results/m2_dev/DEV_RESULTS.md").read_text())